# Gold — dim_date

`silver.monthly_performance` → **`gold.dim_date`**.

One row per month, **2011-08 to 2026-08**. Five columns, all of them used by something.

The calendar is **generated as a month sequence**, not taken from `SELECT DISTINCT` on the
facts. Silver deletes months it cannot trust, and a gap in the facts must never become a gap
in the calendar — otherwise a horizon would quietly measure the wrong number of months.

Expected: **181 rows**.

In [0]:
CREATE TABLE IF NOT EXISTS `index-vs-trust-pipeline`.gold.dim_date (
  month_key    INT    COMMENT 'YYYYMM. The join key: readable, and sorts chronologically',
  month_start  DATE   COMMENT 'Date arithmetic for the horizon windows',
  year         INT,
  month_label  STRING COMMENT 'Mar 2024. Chart axes, so the dashboard formats nothing',
  market_event STRING COMMENT 'Names the few months worth annotating; null otherwise'
)
COMMENT 'Monthly calendar covering the study window, 2011-08 onward';

In [0]:
CREATE OR REPLACE TEMP VIEW gold_stage_date AS
WITH bounds AS (
  SELECT MIN(month_start) AS lo, MAX(month_start) AS hi
  FROM `index-vs-trust-pipeline`.silver.monthly_performance
),
months AS (
  -- Generated, never SELECT DISTINCT: a deleted month must not become a missing month.
  SELECT EXPLODE(SEQUENCE(lo, hi, INTERVAL 1 MONTH)) AS month_start FROM bounds
)
SELECT CAST(DATE_FORMAT(month_start, 'yyyyMM') AS INT) AS month_key,
       month_start,
       YEAR(month_start)                              AS year,
       DATE_FORMAT(month_start, 'MMM yyyy')           AS month_label,
       -- The months worth labelling on a chart. They are also the answer to "how do you
       -- know your cleaning did not delete a real crash?" -- their returns are intact.
       CASE DATE_FORMAT(month_start, 'yyyy-MM')
            WHEN '2020-03' THEN 'COVID crash'
            WHEN '2020-04' THEN 'COVID rebound'
            WHEN '2022-06' THEN 'Inflation bear market'
            WHEN '2024-08' THEN 'August selloff'
       END                                            AS market_event
FROM months;

In [0]:
MERGE INTO `index-vs-trust-pipeline`.gold.dim_date AS t
USING gold_stage_date AS s
   ON t.month_key = s.month_key
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

## Verification

In [0]:
SELECT COUNT(*)                                            AS rows_total,
       MIN(month_key)                                      AS first_month,
       MAX(month_key)                                      AS last_month,
       COUNT(*) - COUNT(DISTINCT month_key)                AS duplicates,
       -- A generated calendar has no holes: the count must equal the span.
       CAST(MONTHS_BETWEEN(MAX(month_start), MIN(month_start)) + 1 AS INT) AS span_in_months,
       SUM(CASE WHEN market_event IS NOT NULL THEN 1 ELSE 0 END) AS labelled_months
FROM `index-vs-trust-pipeline`.gold.dim_date;

Expect **181 / 201108 / 202608 / 0 / 181 / 4**.

`rows_total` and `span_in_months` must be **the same number**. If they differ, the calendar
has a hole in it and every horizon that crosses the hole would be measured over the wrong
number of months.

In [0]:
-- The crash is still in the data. This is the answer to "did your cleaning delete it?"
SELECT d.month_key, d.month_label, d.market_event,
       COUNT(*)                                        AS trusts_with_a_return,
       ROUND(100 * PERCENTILE_APPROX(m.total_return, 0.5), 1) AS median_return_pct
FROM `index-vs-trust-pipeline`.gold.dim_date d
JOIN `index-vs-trust-pipeline`.silver.monthly_performance m ON m.month_key = d.month_key
WHERE d.market_event IS NOT NULL AND m.total_return IS NOT NULL
GROUP BY d.month_key, d.month_label, d.market_event
ORDER BY d.month_key;

Expect **4 rows**, with `2020-03` showing a median around **−19%** across roughly 90 trusts.

That row is worth having on screen: the COVID crash is present, intact, and measured — and
the scale-repair rule flagged **zero** rows in that month.